# Limpieza y análisis exploratorio de datos

Este notebook contiene la revisión inicial de los datos logísticos, su limpieza y un análisis exploratorio para identificar patrones en los envíos de banano.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

df = pd.read_csv('logistica_banano_uraba.csv')

display(df.head())
print('Dimensiones:', df.shape)
display(df.dtypes)
display(df.isnull().sum())

## Limpieza de datos

Se corrigen inconsistencias en los nombres de los municipios y se revisa el formato de las fechas. Los valores faltantes se conservan por ahora para evaluar su tratamiento durante el análisis.

In [ ]:
df['municipio'] = df['municipio'].astype('string').str.strip().str.title()

correcciones = {
    'Apartado': 'Apartadó',
    'Chigorodo': 'Chigorodó'
}

df['municipio'] = df['municipio'].replace(correcciones)
df['fecha_envio'] = pd.to_datetime(df['fecha_envio'], errors='coerce')

print('Municipios después de la limpieza:')
display(df['municipio'].value_counts(dropna=False))

print('Valores faltantes después de la limpieza:')
display(df.isnull().sum())

## Revisión de calidad

Se revisan los valores faltantes, los registros duplicados y las estadísticas generales para conocer el estado del conjunto de datos antes de analizarlo.

In [ ]:
resumen_nulos = pd.DataFrame({
    'Valores faltantes': df.isnull().sum(),
    'Porcentaje (%)': (df.isnull().mean() * 100).round(2)
})

display(resumen_nulos)

print('Registros duplicados:', df.duplicated().sum())

if 'id_viaje' in df.columns:
    print('ID de viaje duplicados:', df['id_viaje'].duplicated().sum())

display(df.describe(include='all'))

## Análisis exploratorio

Se revisan los envíos desde diferentes perspectivas: municipio, turno y tiempo de tránsito. La tasa de rechazo se calcula sobre el total de cajas enviadas, no como un promedio simple de los porcentajes de cada registro.

In [ ]:
total_enviadas = df['cajas_enviadas'].sum()
total_rechazadas = df['cajas_rechazadas'].sum()

tasa_rechazo_general = (
    total_rechazadas / total_enviadas * 100
    if total_enviadas > 0 else 0
)

print(f'Total de cajas enviadas: {total_enviadas:,.0f}')
print(f'Total de cajas rechazadas: {total_rechazadas:,.0f}')
print(f'Tasa de rechazo general: {tasa_rechazo_general:.2f}%')

### Resultados por municipio

In [ ]:
analisis_municipio = df.groupby('municipio', dropna=False).agg(
    viajes=('municipio', 'size'),
    cajas_enviadas=('cajas_enviadas', 'sum'),
    cajas_rechazadas=('cajas_rechazadas', 'sum')
).reset_index()

analisis_municipio['tasa_rechazo_pct'] = (
    analisis_municipio['cajas_rechazadas'] /
    analisis_municipio['cajas_enviadas'].replace(0, np.nan) * 100
).round(2)

display(analisis_municipio.sort_values('tasa_rechazo_pct', ascending=False))

### Resultados por turno

In [ ]:
analisis_turno = df.groupby('turno', dropna=False).agg(
    viajes=('turno', 'size'),
    cajas_enviadas=('cajas_enviadas', 'sum'),
    cajas_rechazadas=('cajas_rechazadas', 'sum')
).reset_index()

analisis_turno['tasa_rechazo_pct'] = (
    analisis_turno['cajas_rechazadas'] /
    analisis_turno['cajas_enviadas'].replace(0, np.nan) * 100
).round(2)

display(analisis_turno)

### Tiempo de tránsito

Para este análisis se excluyen temporalmente los registros sin información de horas de tránsito. Los datos originales se mantienen sin modificar.

In [ ]:
df_transito = df.dropna(subset=['horas_transito']).copy()

df_transito['rango_transito'] = pd.cut(
    df_transito['horas_transito'],
    bins=[-np.inf, 2, 4, 6, np.inf],
    labels=['Hasta 2 horas', 'Más de 2 a 4', 'Más de 4 a 6', 'Más de 6']
)

analisis_transito = df_transito.groupby(
    'rango_transito', observed=False
).agg(
    viajes=('rango_transito', 'size'),
    cajas_enviadas=('cajas_enviadas', 'sum'),
    cajas_rechazadas=('cajas_rechazadas', 'sum')
).reset_index()

analisis_transito['tasa_rechazo_pct'] = (
    analisis_transito['cajas_rechazadas'] /
    analisis_transito['cajas_enviadas'].replace(0, np.nan) * 100
).round(2)

display(analisis_transito)

## Visualización de resultados

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(analisis_municipio['municipio'].astype(str), analisis_municipio['tasa_rechazo_pct'])
plt.title('Tasa de rechazo por municipio')
plt.xlabel('Municipio')
plt.ylabel('Tasa de rechazo (%)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(analisis_transito['rango_transito'].astype(str), analisis_transito['tasa_rechazo_pct'])
plt.title('Tasa de rechazo según tiempo de tránsito')
plt.xlabel('Rango de horas de tránsito')
plt.ylabel('Tasa de rechazo (%)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Exportación

Se guarda el conjunto de datos con las correcciones realizadas. Los valores faltantes de horas de tránsito se conservan para decidir su tratamiento con base en el análisis y no introducir una imputación sin justificación.

In [ ]:
df.to_csv('logistica_limpia.csv', index=False)

print('Archivo exportado: logistica_limpia.csv')
print('Valores faltantes de horas de tránsito:', df['horas_transito'].isnull().sum())
print('Registros totales:', len(df))